In [0]:
import os

VOL_BASE = "/Volumes/aegis_fraud_workspace/finguard/finguard_volume"

# Ensure dirs exist
for sub in ["raw_landing", "_checkpoints/schema_registry", "_checkpoints/bronze", "_checkpoints/silver"]:
    os.makedirs(f"{VOL_BASE}/{sub}", exist_ok=True)

# Write a dummy seed file so Auto Loader finds at least one file immediately
seed_payload = '[{"transaction_id":"INIT_000","user_id":"USR_001","amount":10.0,"merchant":"Init","location":"Hyderabad","timestamp":"2026-01-01 00:00:00"}]'
with open(f"{VOL_BASE}/raw_landing/init_seed.json", "w") as f:
    f.write(seed_payload)

print("Seed file created. Ready to start streams.")

Seed file created. Ready to start streams.


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

landing_path = "/Volumes/aegis_fraud_workspace/finguard/finguard_volume/raw_landing"
schema_tracking = "/Volumes/aegis_fraud_workspace/finguard/finguard_volume/_checkpoints/schema_registry"
chk_bronze = "/Volumes/aegis_fraud_workspace/finguard/finguard_volume/_checkpoints/bronze"

raw_payload_schema = StructType([
    StructField("transaction_id", StringType(), False),
    StructField("user_id", StringType(), False),
    StructField("amount", DoubleType(), False),
    StructField("merchant", StringType(), True),
    StructField("location", StringType(), True),
    StructField("timestamp", StringType(), False)
])

df_raw_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", schema_tracking)
    .schema(raw_payload_schema)
    .load(landing_path)
    .withColumn("event_timestamp", F.to_timestamp("timestamp", "yyyy-MM-dd HH:mm:ss"))
    .withColumn("ingested_at", F.current_timestamp())
)

bronze_writer = (
    df_raw_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", chk_bronze)
    .trigger(availableNow=True)
    .toTable("aegis_fraud_workspace.finguard.bronze_transactions")
)

print(f"[ACTIVE] Bronze stream started: {bronze_writer.id}")

[ACTIVE] Bronze stream started: 02cbeb11-b742-440d-a521-7cdd28b2e0ff


In [0]:
# Check detailed stream status
print("Active status:", bronze_writer.isActive)
print("Status:", bronze_writer.status)
print("Exception details:")
print(bronze_writer.exception())

Active status: True
Status: {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}
Exception details:
None


In [0]:
%sql
SELECT * FROM aegis_fraud_workspace.finguard.bronze_transactions;

transaction_id,user_id,amount,merchant,location,timestamp,event_timestamp,ingested_at
TXN_2206816,USR_008,314.74,Starbucks,Hyderabad,2026-09-11 02:53:30,2026-09-11T02:53:30.000Z,2026-09-11T02:55:40.947Z
TXN_8290322,USR_002,244.11,Amazon Web Services,Hyderabad,2026-09-11 02:53:30,2026-09-11T02:53:30.000Z,2026-09-11T02:55:40.947Z
TXN_7448180,USR_004,8114.13,Starbucks,Singapore,2026-09-11 02:53:30,2026-09-11T02:53:30.000Z,2026-09-11T02:55:40.947Z
TXN_6023592,USR_001,163.4,Apple Store,Hyderabad,2026-09-11 02:53:30,2026-09-11T02:53:30.000Z,2026-09-11T02:55:40.947Z
TXN_9473938,USR_002,250.26,Amazon Web Services,Hyderabad,2026-09-11 02:53:30,2026-09-11T02:53:30.000Z,2026-09-11T02:55:40.947Z
TXN_5721365,USR_003,6348.82,Zara Fashion,Dubai,2026-09-11 02:53:30,2026-09-11T02:53:30.000Z,2026-09-11T02:55:40.947Z
TXN_1807248,USR_005,73.18,Apple Store,Hyderabad,2026-09-11 02:53:25,2026-09-11T02:53:25.000Z,2026-09-11T02:55:40.947Z
TXN_5498485,USR_008,215.94,Amazon Web Services,Hyderabad,2026-09-11 02:53:25,2026-09-11T02:53:25.000Z,2026-09-11T02:55:40.947Z
TXN_4923855,USR_004,220.54,Delta Airlines,Hyderabad,2026-09-11 02:53:25,2026-09-11T02:53:25.000Z,2026-09-11T02:55:40.947Z
TXN_2289943,USR_002,209.82,Starbucks,Hyderabad,2026-09-11 02:53:25,2026-09-11T02:53:25.000Z,2026-09-11T02:55:40.947Z


In [0]:
%sql
CREATE TABLE IF NOT EXISTS aegis_fraud_workspace.finguard.dim_customers AS
SELECT *
FROM aegis_fraud_workspace.default.dim_customers;

num_affected_rows,num_inserted_rows


In [0]:
display(spark.table("aegis_fraud_workspace.finguard.dim_customers").limit(5))

user_id,customer_name,home_city,daily_limit,segment
USR_001,Aarav Sharma,Hyderabad,5000.0,TIER_1
USR_002,Priya Nair,Bengaluru,2500.0,TIER_2
USR_003,Rohan Mehta,Mumbai,7500.0,TIER_1
USR_004,Sneha Rao,Hyderabad,1500.0,TIER_3
USR_005,Vikram Verma,London,10000.0,PRIVATE_WEALTH


In [0]:
from pyspark.sql import functions as F

chk_silver = "/Volumes/aegis_fraud_workspace/finguard/finguard_volume/_checkpoints/silver"

df_bronze = spark.readStream.table("aegis_fraud_workspace.finguard.bronze_transactions")
df_cust_dim = spark.table("aegis_fraud_workspace.finguard.dim_customers")

df_silver_enriched = (
    df_bronze.join(
        F.broadcast(df_cust_dim),
        on="user_id",
        how="inner"
    )
    .withColumn("is_out_of_home", F.when(F.col("location") != F.col("home_city"), 1.0).otherwise(0.0))
    .withColumn("limit_ratio", F.round(F.col("amount") / F.col("daily_limit"), 4))
    .select(
        "transaction_id",
        "user_id",
        "customer_name",
        "segment",
        "amount",
        "daily_limit",
        "limit_ratio",
        "home_city",
        "location",
        "is_out_of_home",
        "event_timestamp",
        "ingested_at"
    )
)

silver_writer = (
    df_silver_enriched.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", chk_silver)
    .trigger(availableNow=True)
    .toTable("aegis_fraud_workspace.finguard.silver_transactions")
)

silver_writer.awaitTermination()
print(f"[SUCCESS] Silver batch enriched. Total records: {spark.table('aegis_fraud_workspace.finguard.silver_transactions').count()}")

[SUCCESS] Silver batch enriched. Total records: 90


In [0]:
%sql
SELECT * FROM aegis_fraud_workspace.finguard.silver_transactions;

transaction_id,user_id,customer_name,segment,amount,daily_limit,limit_ratio,home_city,location,is_out_of_home,event_timestamp,ingested_at
TXN_2206816,USR_008,Neha Kulkarni,TIER_3,314.74,2000.0,0.1574,Pune,Hyderabad,1.0,2026-09-11T02:53:30.000Z,2026-09-11T02:55:40.947Z
TXN_8290322,USR_002,Priya Nair,TIER_2,244.11,2500.0,0.0976,Bengaluru,Hyderabad,1.0,2026-09-11T02:53:30.000Z,2026-09-11T02:55:40.947Z
TXN_7448180,USR_004,Sneha Rao,TIER_3,8114.13,1500.0,5.4094,Hyderabad,Singapore,1.0,2026-09-11T02:53:30.000Z,2026-09-11T02:55:40.947Z
TXN_6023592,USR_001,Aarav Sharma,TIER_1,163.4,5000.0,0.0327,Hyderabad,Hyderabad,0.0,2026-09-11T02:53:30.000Z,2026-09-11T02:55:40.947Z
TXN_9473938,USR_002,Priya Nair,TIER_2,250.26,2500.0,0.1001,Bengaluru,Hyderabad,1.0,2026-09-11T02:53:30.000Z,2026-09-11T02:55:40.947Z
TXN_5721365,USR_003,Rohan Mehta,TIER_1,6348.82,7500.0,0.8465,Mumbai,Dubai,1.0,2026-09-11T02:53:30.000Z,2026-09-11T02:55:40.947Z
TXN_1807248,USR_005,Vikram Verma,PRIVATE_WEALTH,73.18,10000.0,0.0073,London,Hyderabad,1.0,2026-09-11T02:53:25.000Z,2026-09-11T02:55:40.947Z
TXN_5498485,USR_008,Neha Kulkarni,TIER_3,215.94,2000.0,0.108,Pune,Hyderabad,1.0,2026-09-11T02:53:25.000Z,2026-09-11T02:55:40.947Z
TXN_4923855,USR_004,Sneha Rao,TIER_3,220.54,1500.0,0.147,Hyderabad,Hyderabad,0.0,2026-09-11T02:53:25.000Z,2026-09-11T02:55:40.947Z
TXN_2289943,USR_002,Priya Nair,TIER_2,209.82,2500.0,0.0839,Bengaluru,Hyderabad,1.0,2026-09-11T02:53:25.000Z,2026-09-11T02:55:40.947Z
